In [1]:
!pip install transformers torch

In [33]:
import nbformat

nb = nbformat.read("Task3.ipynb", as_version=4)

# Remove problematic metadata
nb.metadata.pop("widgets", None)

nbformat.write(nb, "clean_notebook.ipynb")

In [19]:
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"

from transformers.utils import logging
logging.set_verbosity_error()



In [30]:
import os
os.listdir()

['.config', 'sample_data']

In [20]:

from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging

logging.set_verbosity_error()

In [21]:
!pip install transformers torch --quiet

In [22]:
# Standard library imports
import warnings
warnings.filterwarnings('ignore')
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model with automatic dtype and device placement
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()

print("Model loaded successfully.")
print("Device map:", model.hf_device_map if hasattr(model, "hf_device_map") else "Single device")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully.
Device map: Single device


In [23]:
def generate_reply(chat_history, user_message, max_new_tokens=256):

    # Add the new user message to the history
    chat_history.append({"role": "user", "content": user_message})
    prompt_text = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Tokenize
    inputs = tokenizer(
        [prompt_text],
        return_tensors="pt"
    ).to(model.device)

    # Generate model output
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05
        )

    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]

    assistant_reply = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    # Add assistant reply to history
    chat_history.append({"role": "assistant", "content": assistant_reply})
    return chat_history, assistant_reply

In [25]:
def run_chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

    # Initialize chat history with a system message to set behavior
    chat_history = [
        {
            "role": "system",
            "content": "You are a helpful, concise AI assistant. Answer clearly and politely."
        }
    ]
    while True:
        user_input = input("User: ").strip()

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a great day.")
            break

        # Skip empty input
        if not user_input:
            print("Chatbot: Please type something to provide a  respond.")
            continue

        # Generate reply from the model
        chat_history, assistant_reply = generate_reply(chat_history, user_input)

        # fallback
        if assistant_reply == "":
            assistant_reply = "I'm not sure how to respond to that yet, but I'm learning."
        print("Chatbot:", assistant_reply)

run_chatbot()


Chatbot: Hello! I am your AI assistant. How can I help you today?
User: hello
Chatbot: Hello! How can I assist you today?
User: what is programming?
Chatbot: Programming is the process of writing instructions, usually in a high-level programming language, for a computer to execute and carry out specific tasks. It involves using code to create software applications, websites, games, or other digital products. Programming languages allow programmers to express algorithms and logic in a way that computers can understand and execute.
User: who invented python?
Chatbot: Python was invented by Guido van Rossum in 1989 while he was at CWI (Centrum Wiskunde & Informatica) in the Netherlands. He started working on it in February 1991 as a successor to the ABC programming language. The first public release of Python was in 1994, and since then, it has become one of the most popular programming languages worldwide.
User: thank you
Chatbot: You're welcome! If you have any more questions, feel free